In [0]:
# Query for the enriched reviews dataset stored in Delta Table
from pyspark.sql.functions import col, rank, count, when, sum, lit, sum_distinct
from pyspark.sql.window import Window
from pyspark.sql import functions as F

df_enriched = spark.table("workspace.default.sephora_reviews_enriched_emily")
display(df_enriched)

In [0]:
#Split data into train test set

interactions_train_df, interactions_test_df = df_enriched.randomSplit([0.8,0.2], seed=123)

In [0]:
interactions_train_df = interactions_train_df.withColumn("rating", interactions_train_df["rating"].cast("int"))
interactions_test_df = interactions_test_df.withColumn("rating", interactions_test_df["rating"].cast("int"))
display(interactions_train_df)
display(interactions_test_df)

In [0]:
print(interactions_train_df.count())
print(interactions_test_df.count())

In [0]:
#Get most popular products in the train data
#product_rating_df = interactions_train_df.groupBy("brand_name","product_name","product_id").agg({"rating": "sum"}).orderBy("sum(rating)", ascending=False)

#Get most popular products in the test data
#product_rating_df_test = interactions_test_df.groupBy("brand_name","product_name","product_id").agg({"rating": "sum"}).orderBy("sum(rating)", ascending=False)

#display(product_rating_df)
#display(product_rating_df_test)

In [0]:
# Show the frequency of each rating for each product_id, including brand_name and product_name, sorted by highest rating frequency
rating_freq_df = interactions_train_df.groupBy("product_id", "brand_name", "product_name", "rating").count().orderBy("count", ascending=False)
rating_freq_df = rating_freq_df.withColumn("count", rating_freq_df["count"].cast("int"))
display(rating_freq_df)

In [0]:
# Show the frequency of each rating for each product_id, including brand_name and product_name, sorted by highest rating frequency
rating_freq_df_all = interactions_train_df.groupBy("product_id", "brand_name", "product_name").count().orderBy("count", ascending=False)
rating_freq_df_all = rating_freq_df_all.withColumn("count", rating_freq_df_all["count"].cast("int"))
display(rating_freq_df_all)

In [0]:
unique_product_count = interactions_train_df.select("product_id").distinct().count()
print(unique_product_count)

In [0]:
row_count = interactions_train_df.count()
print(row_count)

In [0]:
unique_users = interactions_train_df.select("author_id").distinct().count()
print(unique_users)

In [0]:
import pyspark.pandas as ps
import matplotlib.pyplot as plt

pd_rating = rating_freq_df_all.toPandas()

# Get counts of ratings > 4 for each top 10 product_name
pd_rating_top10 = pd_rating.sort_values(by='count', ascending=False).head(10)
pd_rating_top10 = pd_rating_top10.sort_values(by='count', ascending=True)
top10_product_names = pd_rating_top10['product_name'].tolist()
pd_rating_gt4 = rating_freq_df.filter(rating_freq_df['rating'] >= 4).toPandas()
pd_rating_gt4_top10 = pd_rating_gt4[pd_rating_gt4['product_name'].isin(top10_product_names)]
gt4_counts = pd_rating_gt4_top10.groupby('product_name')['count'].sum().reindex(pd_rating_top10['product_name']).fillna(0)

# Calculate percentages
total_rows = pd_rating['count'].sum()
total_counts = pd_rating_top10['count']
total_percent = (total_counts / total_rows * 100).round(2)
gt4_percent = (gt4_counts / total_rows * 100).round(2)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))  # Increased horizontal size
bar_width = 0.4
indices = range(len(pd_rating_top10))

ax.barh(indices, pd_rating_top10['count'], bar_width, label='Total Count')
ax.barh([i + bar_width for i in indices], gt4_counts.values, bar_width, label='Count (Rating >= 4)')

ax.set_yticks([i + bar_width / 2 for i in indices])
ax.set_yticklabels(pd_rating_top10['product_name'])
plt.xlabel("Count")
plt.ylabel("Product name")
plt.title("Top 10 Product Names by Count (Descending Order)")
plt.legend()
plt.tight_layout()

# Annotate bars with counts and percentages
for i, (count, percent) in enumerate(zip(total_counts, total_percent)):
    ax.text(count + 5, i, f"{count} ({percent}%)", va='center', fontsize=10)
for i, (count, percent) in enumerate(zip(gt4_counts.values, gt4_percent)):
    ax.text(count + 5, i + bar_width, f"{int(count)} ({percent}%)", va='center', fontsize=10)

plt.show()

In [0]:
import matplotlib.pyplot as plt

# Filter for ratings 4 and 5
rating_4_5_df = rating_freq_df.filter(rating_freq_df['rating']>=4)

# Group by product_name and sum counts for ratings 4 and 5
top_products_4_5 = rating_4_5_df.groupBy("product_name","product_id").agg({"count": "sum"}).orderBy("sum(count)", ascending=False).limit(10)

# Convert to pandas for plotting
pd_top_products_4_5 = top_products_4_5.toPandas().sort_values(by="sum(count)", ascending=True)

# Calculate total rows for percentage
total_rows = interactions_train_df.count()
percentages = (pd_top_products_4_5["sum(count)"] / total_rows * 100).round(2)

# Plot horizontal bar chart
plt.figure(figsize=(12, 6))
plt.barh(pd_top_products_4_5["product_name"], pd_top_products_4_5["sum(count)"], color="skyblue")
plt.xlabel("Count of Ratings 4 & 5")
plt.ylabel("Product Name")
plt.title("Top 10 Product Names by Count of Ratings 4 & 5")
plt.tight_layout()

# Annotate bars with counts and percentages
for i, (count, percent) in enumerate(zip(pd_top_products_4_5["sum(count)"], percentages)):
    plt.text(count + 5, i, f"{count} ({percent}%)", va='center', fontsize=10)

plt.show()

In [0]:
# Calculate total sum of ratings for each product in pd_top_products_4_5 using ratings*count
pd_rating_4_5 = rating_4_5_df.toPandas()
top10_names = pd_top_products_4_5["product_name"].tolist()
pd_rating_4_5_top10 = pd_rating_4_5[pd_rating_4_5["product_name"].isin(top10_names)]

# Compute sum of ratings*count for each product
pd_rating_4_5_top10["rating_sum"] = pd_rating_4_5_top10["rating"] * pd_rating_4_5_top10["count"]
rating_sum_by_product = pd_rating_4_5_top10.groupby("product_name")["rating_sum"].sum().reindex(top10_names).fillna(0)
#top10_popular = pd_rating_4_5_top10.groupby(["product_name","product_id"]).sum().sort_values(by='rating_sum',ascending=False)
top10_popular = pd_rating_4_5_top10.groupby(["product_name","product_id","brand_name"]).sum().sort_values(by='rating_sum',ascending=False)
# Sort by descending order
rating_sum_by_product = rating_sum_by_product.sort_values(ascending=True)

# Plot bar chart
plt.figure(figsize=(12, 6))
plt.barh(rating_sum_by_product.index, rating_sum_by_product.values, color="orange")
plt.xlabel("Total Sum of Ratings (ratings*count)")
plt.ylabel("Product Name")
plt.title("Total Sum of Ratings for Top 10 Products (Ratings 4 & 5)")
plt.tight_layout()

# Annotate bars
for i, value in enumerate(rating_sum_by_product.values):
    plt.text(value + 5, i, f"{int(value)}", va='center', fontsize=10)

plt.show()

In [0]:
top10_popular

In [0]:
top10_popular_df = top10_popular.reset_index()[['product_name', 'product_id', 'brand_name']]
top10_popular_df = spark.createDataFrame(top10_popular_df)
display(top10_popular_df)

In [0]:
top10_product_ids = top10_popular.index.get_level_values('product_id').tolist()
print(top10_product_ids)

In [0]:
# Calculate each user's avg rating for determining relevant products
user_train_avg = (
    interactions_train_df
    .groupBy("author_id")
    .agg(F.avg("rating").alias("user_train_avg"))
)

interactions_train_df = (
    interactions_train_df
    .join(user_train_avg, on="author_id", how="left")
)

display(interactions_train_df)

user_test_avg = (
    interactions_test_df
    .groupBy("author_id")
    .agg(F.avg("rating").alias("user_test_avg"))
)

interactions_test_df = (
    interactions_test_df
    .join(user_test_avg, on="author_id", how="left")
)

display(interactions_test_df)

In [0]:
# Get top 10 product_ids by popularity (train)
#top_products_df = product_rating_df.select("product_id").limit(10)
#top_products_df = top10_product_ids
#top_product_ids = [row.product_id for row in top_products_df.collect()]

# test
#top_products_df_test = product_rating_df_test.select("product_id").limit(10)
#top_product_ids_test = [row.product_id for row in top_products_df_test.collect()]

# Add 'hit' column: 1 if product_id in top 10, else 0
# Define the window specification to partition by 'Category'
window_spec = Window.partitionBy("author_id")

# Add a new column with relevant items count using withColumn and count over the window
interactions_train_df = interactions_train_df.withColumn(
    "relevant_items",
    F.count((when(col("rating") >= 4.0, True))).over(window_spec)
)

# Filter out users with 0 relevant items before calculating evaluation metrics
interactions_train_with_relevant = interactions_train_df.filter(col("relevant_items") > 0)

interactions_train_with_relevant = interactions_train_with_relevant.withColumn(
    "hit",
    when((interactions_train_with_relevant["product_id"].isin(top10_product_ids) & (interactions_train_with_relevant["rating"] >= 4.0)), 1).otherwise(0)
)

#interactions_test_df = interactions_test_df.withColumn(
#    "hit",
#    when(interactions_test_df["product_id"].isin(top10_product_ids_test), 1).otherwise(0)
#)
#display(interactions_train_df)
#display(interactions_test_df)

In [0]:
display(interactions_train_df)

In [0]:
K = 10
user_precision = interactions_train_with_relevant.groupBy("author_id").agg(
    (sum("hit") / lit(K)).alias("precision_at_k")
)

user_recall = interactions_train_with_relevant.groupBy("author_id").agg(
    (sum("hit") / sum_distinct("relevant_items")).alias("recall_at_k")
)

# Final Average Precision
mean_precision = user_precision.agg({"precision_at_k": "mean"}).collect()[0][0]
print(f"Mean Precision@{K}: {mean_precision}")

mean_recall = user_recall.agg({"recall_at_k": "mean"}).collect()[0][0]
print(f"Mean Recall@{K}: {mean_recall}")

In [0]:
# Add a new column with relevant items count using withColumn and count over the window
interactions_test_df = interactions_test_df.withColumn(
    "relevant_items",
    F.count((when(col("rating") >= 4.0, True))).over(window_spec)
)

# Filter out users with 0 relevant items before calculating evaluation metrics
interactions_test_with_relevant = interactions_test_df.filter(col("relevant_items") > 0)

interactions_test_with_relevant = interactions_test_with_relevant.withColumn(
    "hit",
    when((interactions_test_with_relevant["product_id"].isin(top10_product_ids) & (interactions_test_with_relevant["rating"] >= 4.0)), 1).otherwise(0)
)

In [0]:
display(interactions_test_with_relevant)

In [0]:
#Evaluate on test set
user_precision_test = interactions_test_with_relevant.groupBy("author_id").agg(
    (sum("hit") / lit(K)).alias("precision_at_k_test")
)

user_recall_test = interactions_test_with_relevant.groupBy("author_id").agg(
    (sum("hit") / sum_distinct("relevant_items")).alias("recall_at_k_test")
)

# Final Average Precision
mean_precision_test = user_precision_test.agg({"precision_at_k_test": "mean"}).collect()[0][0]
print(f"Mean Precision@{K} Test: {mean_precision_test}")

mean_recall_test = user_recall_test.agg({"recall_at_k_test": "mean"}).collect()[0][0]
print(f"Mean Recall@{K} Test: {mean_recall_test}")

In [0]:
# Aggregate test items per user
interactions_high_ratings = interactions_train_with_relevant.filter(col("rating") >= 4.0)
actual_labels = interactions_high_ratings.groupBy("author_id") \
    .agg(F.collect_set("product_id").alias("actual_items"))

# Create recommendations for all users (cross join or mapping)
# Since popular items are same, we just join them to test users
recommendations = actual_labels.select("author_id") \
    .withColumn("predicted_items", F.array([F.lit(i) for i in top10_product_ids]))


In [0]:
# Join recommendations with actuals
eval_df = recommendations.join(actual_labels, "author_id", "inner")

# Calculate Average Precision (AP) using UDF or SQL functions
# For each user, AP = (1 / |Actuals|) * sum(Precision@k * relevance@k)
def calculate_ap(predicted, actual):
    if not actual:
        return 0.0
    
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p in actual:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    
    return score / min(len(actual), len(predicted))

# Register the function
from pyspark.sql.types import DoubleType
calculate_ap_udf = F.udf(calculate_ap, DoubleType())

# Calculate AP for each user
eval_df = eval_df.withColumn("AP", calculate_ap_udf(F.col("predicted_items"), F.col("actual_items")))

# Calculate Mean AP
map_score = eval_df.agg(F.mean("AP")).collect()[0][0]
print(f"MAP@{K}: {map_score}")


In [0]:
# Repeat for Test Set
interactions_high_ratings_test = interactions_test_with_relevant.filter(col("rating") >= 4.0)
actual_labels_test = interactions_high_ratings_test.groupBy("author_id") \
    .agg(F.collect_set("product_id").alias("actual_items_test"))

# Join recommendations with actuals
eval_df_test = recommendations.join(actual_labels_test, "author_id", "inner")

# Calculate AP for each user
eval_df_test = eval_df_test.withColumn("AP_test", calculate_ap_udf(F.col("predicted_items"), F.col("actual_items_test")))

# Calculate Mean AP
map_score_test = eval_df_test.agg(F.mean("AP_test")).collect()[0][0]
print(f"MAP@{K} Test: {map_score_test}")


In [0]:
'''
#  Temp added
# Create predictions dataframe for Popularity Recommender
# For each user in test set, recommend the top 10 popular items with prediction score = popularity rank

# Get popularity scores for top10 items from training data
item_popularity_scores = (
    interactions_train_df
    .filter(F.col("product_id").isin(top10_product_ids))
    .groupBy("product_id")
    .agg(F.count("*").alias("popularity_score"))
)

# Create predictions: cross join test users with top 10 popular items
test_users = interactions_test_df.select("author_id").distinct()
popularity_predictions = (
    test_users
    .crossJoin(item_popularity_scores)
    .select(
        F.col("author_id").alias("user_id"),
        F.col("product_id").alias("item_id"),
        F.col("popularity_score").alias("prediction")
    )
)

# Join with actual test ratings
df_pred = (
    popularity_predictions
    .join(
        interactions_test_df.select(
            F.col("author_id").alias("user_id"),
            F.col("product_id").alias("item_id"),
            F.col("rating").alias("actual_rating")
        ),
        on=["user_id", "item_id"],
        how="inner"
    )
    .withColumn("support", F.lit(1))  # Add support column (constant for popularity)
)

print(f"Prediction rows for evaluation: {df_pred.count()}")
display(df_pred.limit(10))

# Use training data for catalog calculations
ratings = interactions_train_df.select(
    F.col("author_id").alias("user_id"),
    F.col("product_id").alias("item_id"),
    F.col("rating")
)

# Comprehensive evaluation loop for varying K from 1-10
results = []
for K in range(1, 11):
    rank_window = Window.partitionBy("user_id").orderBy(F.desc("prediction"), F.desc("support"), F.asc("item_id"))
    ranked_pred = (
        df_pred
        .withColumn("rank", F.row_number().over(rank_window))
        .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
    )
    topk = ranked_pred.filter(F.col("rank") <= K)
    user_relevant_counts = (
        df_pred
        .withColumn("relevant", F.when(F.col("actual_rating") >= 4.0, 1).otherwise(0))
        .groupBy("user_id")
        .agg(F.sum("relevant").alias("n_relevant"))
    )
    per_user_hits = (
        topk.groupBy("user_id")
        .agg(F.sum("relevant").alias("hits"))
        .join(user_relevant_counts, on="user_id", how="left")
        .fillna(0, subset=["n_relevant"])
        .withColumn("precision_at_k", F.col("hits") / F.lit(K))
        .withColumn(
            "recall_at_k",
            F.when(F.col("n_relevant") > 0, F.col("hits") / F.col("n_relevant")).otherwise(F.lit(0.0))
        )
    )
    ap_base = (
        topk
        .withColumn("cum_relevant", F.sum("relevant").over(rank_window.rowsBetween(Window.unboundedPreceding, 0)))
        .withColumn(
            "precision_at_rank",
            F.when(F.col("relevant") == 1, F.col("cum_relevant") / F.col("rank")).otherwise(F.lit(0.0))
        )
        .groupBy("user_id")
        .agg(F.sum("precision_at_rank").alias("sum_precisions"))
        .join(user_relevant_counts, on="user_id", how="left")
        .fillna(0, subset=["n_relevant"])
        .withColumn(
            "ap_at_k",
            F.when(F.col("n_relevant") > 0, F.col("sum_precisions") / F.least(F.col("n_relevant"), F.lit(K)))
             .otherwise(F.lit(0.0))
        )
    )
    precision_at_k = per_user_hits.agg(F.avg("precision_at_k")).collect()[0][0]
    recall_at_k = per_user_hits.agg(F.avg("recall_at_k")).collect()[0][0]
    map_at_k = ap_base.agg(F.avg("ap_at_k")).collect()[0][0]
    dcg_base = (
        topk
        .withColumn(
            "dcg_component",
            F.when(
                (F.col("relevant") == 1) & (F.col("rank") > 0),
                1 / F.log2(F.col("rank") + 1)
            ).otherwise(F.lit(0.0))
        )
        .groupBy("user_id")
        .agg(F.sum("dcg_component").alias("dcg"))
    )
    ideal_dcg = (
        user_relevant_counts
        .withColumn(
            "ideal_dcg",
            F.expr(f"""
            aggregate(
                sequence(1, greatest(least(n_relevant,{K}),1)),
                0D,
                (acc,x) -> acc + 1/log2(x+1)
            )
            """)
        )
    )
    ndcg_df = (
        dcg_base
        .join(ideal_dcg, on="user_id", how="left")
        .withColumn(
            "ndcg_at_k",
            F.when(
                (F.col("ideal_dcg").isNotNull()) & (F.col("ideal_dcg") > 0),
                F.col("dcg") / F.col("ideal_dcg")
            ).otherwise(F.lit(0.0))
        )
    )
    ndcg_at_k = ndcg_df.select(F.avg("ndcg_at_k")).collect()[0][0]
    total_catalog_items = ratings.select("item_id").distinct().count()
    catalogue_coverage = (
        topk.select("item_id").distinct().count() /
        total_catalog_items
        if total_catalog_items > 0 else 0.0
    )
    item_popularity = ratings.groupBy("item_id").agg(F.count("*").alias("popularity"))
    pop_threshold = item_popularity.approxQuantile("popularity", [0.8], 0.01)[0] if item_popularity.count() > 0 else 0
    serendipity = (
        topk.join(item_popularity, on="item_id", how="left")
        .withColumn("unexpected", F.when(F.col("popularity") <= pop_threshold, 1).otherwise(0))
        .withColumn("serendipitous", F.when((F.col("relevant") == 1) & (F.col("unexpected") == 1), 1).otherwise(0))
        .groupBy("user_id")
        .agg((F.sum("serendipitous") / F.lit(K)).alias("serendipity"))
    )
    serendipity_score = serendipity.agg(F.avg("serendipity")).collect()[0][0]
    user_lists = topk.groupBy("user_id").agg(F.collect_set("item_id").alias("rec_list"))
    user_pairs = (
        user_lists.alias("a")
        .crossJoin(user_lists.alias("b"))
        .filter(F.col("a.user_id") < F.col("b.user_id"))
        .select(
            F.size(F.array_intersect(F.col("a.rec_list"), F.col("b.rec_list"))).alias("intersect_size"),
            F.size(F.array_union(F.col("a.rec_list"), F.col("b.rec_list"))).alias("union_size")
        )
        .withColumn(
            "inter_user_diversity",
            F.when(F.col("union_size") > 0, 1 - (F.col("intersect_size") / F.col("union_size"))).otherwise(F.lit(0.0))
        )
    )
    inter_user_diversity = user_pairs.agg(F.avg("inter_user_diversity")).collect()[0][0]
    random_hit_rate = df_pred.select(F.avg(F.when(F.col("actual_rating") >= 4.0, 1.0).otherwise(0.0))).collect()[0][0]
    lift_over_random = precision_at_k / random_hit_rate if random_hit_rate and random_hit_rate > 0 else None
    
    print(f"K={K}: Precision@K={precision_at_k:.4f}, Recall@K={recall_at_k:.4f}, MAP@K={map_at_k:.4f}")
    
    results.append((K, float(precision_at_k), float(recall_at_k), float(map_at_k), float(ndcg_at_k),
                   float(catalogue_coverage), float(lift_over_random), float(serendipity_score), float(inter_user_diversity)))

metrics_df_varying_k_popularity = spark.createDataFrame(
    results,
    ["K", "Precision@K", "Recall@K", "MAP@K", "NDCG@K", "Catalogue_coverage", "Lift_over_random", "Serendipity", "Inter_user_diversity"]
)
display(metrics_df_varying_k_popularity)
'''

In [0]:
'''
# Temp added
# Save popularity metrics to table for dashboard comparison
metrics_df_varying_k_popularity.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.popularity_metrics_varying_k"
)

print("Saved table: workspace.default.popularity_metrics_varying_k")
'''

In [0]:
#Calculate random hit rate
random_hit_rate = interactions_train_with_relevant.select(
    F.avg(F.when(F.col("rating") >= 4.0, 1.0).otherwise(0.0))
).collect()[0][0]

random_hit_rate_test = interactions_test_with_relevant.select(
    F.avg(F.when(F.col("rating") >= 4.0, 1.0).otherwise(0.0))
).collect()[0][0]

lift = mean_precision/random_hit_rate
lift_test = mean_precision_test/random_hit_rate_test

#Calculate coverage

total_catalog_items = df_enriched.select("product_id").distinct().count()
catalogue_coverage = (
    len(set(top10_product_ids)) /
    total_catalog_items
    if total_catalog_items > 0 else 0.0
)

rank_window = Window.partitionBy("author_id").orderBy(F.desc("rating"), F.asc("product_id"))

ranked_pred_train = (
    interactions_train_with_relevant
    .withColumn("rank", F.row_number().over(rank_window))
)

topk_train = ranked_pred_train.filter(F.col("rank") <= K)

user_relevant_counts_train = (
    interactions_train_with_relevant
    .groupBy("author_id")
    .agg(F.sum("hit").alias("n_relevant"))
)

# NDCG@K (SAFE VERSION)
dcg_base_train = (
    topk_train
    .withColumn(
        "dcg_component",
        F.when(
            (F.col("hit") == 1) & (F.col("rank") > 0),
            1 / F.log2(F.col("rank") + 1)
        ).otherwise(F.lit(0.0))
    )
    .groupBy("author_id")
    .agg(F.sum("dcg_component").alias("dcg"))
)

ideal_dcg_train = (
    user_relevant_counts_train
    .withColumn(
        "ideal_dcg",
        F.expr(f"""
        aggregate(
            sequence(1, greatest(least(n_relevant,{K}),1)),
            0D,
            (acc,x) -> acc + 1/log2(x+1)
        )
        """)
    )
)

ndcg_df_train = (
    dcg_base_train
    .join(ideal_dcg_train, on="author_id", how="left")
    .withColumn(
        "ndcg_at_k",
        F.when(
            (F.col("ideal_dcg").isNotNull()) & (F.col("ideal_dcg") > 0),
            F.col("dcg") / F.col("ideal_dcg")
        ).otherwise(F.lit(0.0))
    )
)

ndcg_at_k_train = ndcg_df_train.select(F.avg("ndcg_at_k")).collect()[0][0]

ranked_pred_test = (
    interactions_test_with_relevant
    .withColumn("rank", F.row_number().over(rank_window))
)

topk_test = ranked_pred_test.filter(F.col("rank") <= K)

user_relevant_counts_test = (
    interactions_test_with_relevant
    .groupBy("author_id")
    .agg(F.sum("hit").alias("n_relevant"))
)

# NDCG@K (SAFE VERSION)
dcg_base_test = (
    topk_test
    .withColumn(
        "dcg_component",
        F.when(
            (F.col("hit") == 1) & (F.col("rank") > 0),
            1 / F.log2(F.col("rank") + 1)
        ).otherwise(F.lit(0.0))
    )
    .groupBy("author_id")
    .agg(F.sum("dcg_component").alias("dcg"))
)

ideal_dcg_test = (
    user_relevant_counts_test
    .withColumn(
        "ideal_dcg",
        F.expr(f"""
        aggregate(
            sequence(1, greatest(least(n_relevant,{K}),1)),
            0D,
            (acc,x) -> acc + 1/log2(x+1)
        )
        """)
    )
)

ndcg_df_test = (
    dcg_base_test
    .join(ideal_dcg_test, on="author_id", how="left")
    .withColumn(
        "ndcg_at_k",
        F.when(
            (F.col("ideal_dcg").isNotNull()) & (F.col("ideal_dcg") > 0),
            F.col("dcg") / F.col("ideal_dcg")
        ).otherwise(F.lit(0.0))
    )
)

ndcg_at_k_test = ndcg_df_test.select(F.avg("ndcg_at_k")).collect()[0][0]

print(f"Training Lift: {lift}")
print(f"Test Lift: {lift_test}")
print(f"Catalogue Coverage: {catalogue_coverage}")
print(f"Training NDCG: {ndcg_at_k_train}")
print(f"Test NDCG: {ndcg_at_k_test}")

In [0]:
#SAVE DELTA TABLE TO WORKSPACE. THIS IS TABLE WE WILL USE FOR TOP RECOMMENDATIONS
#top10_popular_df.write \
#    .format("delta") \
#    .mode("overwrite") \
#    .option("overwriteSchema", "true") \
#    .saveAsTable("workspace.default.topk_popular_items")
display(top10_popular_df)

In [0]:
#Check saved data
topk_popular_items = spark.table("workspace.default.topk_popular_items")
display(topk_popular_items)

In [0]:
metric_data = [
{"model_name": "Popularity",
 "dataset": "Test",
 "precision": (float(mean_precision_test) if mean_precision_test is not None else None),
 "recall": (float(mean_recall_test) if mean_recall_test is not None else None),
 "map": (float(map_score_test) if map_score_test is not None else None),
 "ndcg": (float(ndcg_at_k_test) if ndcg_at_k_test is not None else None),
 "coverage": (float(catalogue_coverage) if catalogue_coverage is not None else None),
 "lift": (float(lift_test) if lift_test is not None else None),
 "k_value": K
}
]

metric_data_train = [
{"model_name": "Popularity",
 "dataset": "Train",
 "precision": (float(mean_precision) if mean_precision is not None else None),
 "recall": (float(mean_recall) if mean_recall is not None else None),
 "map": (float(map_score) if map_score is not None else None),
 "ndcg": (float(ndcg_at_k_train) if ndcg_at_k_train is not None else None),
 "coverage": (float(catalogue_coverage) if catalogue_coverage is not None else None),
 "lift": (float(lift) if lift is not None else None),
 "k_value": K
}
]

model_metrics = spark.createDataFrame(metric_data).withColumn("timestamp", F.current_timestamp())
model_metrics_train = spark.createDataFrame(metric_data_train).withColumn("timestamp", F.current_timestamp())
model_metrics.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.model_metrics")
model_metrics_train.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.default.model_metrics")

model_eval = spark.table("workspace.default.model_metrics")
display(model_eval)


In [0]:
# Recommend the more popular items that the user hasn't seen yet.
#topn = 10
#author_id = "2206599853"
#filter_author = interactions_train_df.filter(interactions_train_df["author_id"] == author_id)
# select_products = filter_author.select("product_id").distinct().sort("product_id")
#recommendations_df = product_rating_df[~product_rating_df['product_id'].isin(select_products)] \
#                               .sort("sum(rating)", ascending = False) \
#                               .head(topn)
#display(recommendations_df)